# Service 1 — Étape 2 : évaluation du modèle de base (AVANT fine-tuning)

On pose les **109 questions de test** à Qwen-2.5-7B-Instruct **tel quel** (aucun entraînement).
Les scores obtenus sont la **référence** : à l'étape 4, on refera exactement la même chose avec le modèle fine-tuné.

| Type de question | Nombre | Mesure |
|---|---|---|
| Service 3 (JSON) | 40 | JSON valide / complet, résultat, cas limite et fiabilité justes |
| MedMCQA (QCM) | 50 | Bonne lettre |
| MedQuAD (questions patients) | 19 | ROUGE-L (ressemblance avec la réponse de référence) |

**Avant de lancer :** *Exécution → Modifier le type d'exécution → GPU T4*, puis *Exécution → Tout exécuter*.
Durée : environ 30 à 45 min (le modèle écrit ses réponses une par une). Les réponses sont sauvegardées au fur et à mesure sur le Drive : si Colab se déconnecte, relancer reprend où on en était.

## 1. Drive, code et chemins

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE = "/content/drive/MyDrive/cardioagent-ops"
REPO_URL = "https://github.com/yasssmine2003/CardioAgent-Ops.git"
REPO_DIR = "/content/CardioAgent-Ops"
if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}/llm_finetuning

EVAL_DIR = f"{DRIVE}/llm/eval"
os.makedirs(EVAL_DIR, exist_ok=True)
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE}/mlruns"
print("Test présent :", os.path.exists("data/final/test.jsonl"))

## 2. Installation (Unsloth, MLflow, ROUGE)
Les avertissements rouges de pip sur des conflits de versions sont habituels sur Colab.

In [ ]:
!pip install -q unsloth "mlflow>=2.17,<3" rouge_score

## 3. Chargement du modèle de base en 4 bits
Version déjà quantifiée en 4 bits par Unsloth : environ 5,5 Go à télécharger, tient dans le GPU T4 (16 Go).

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"   # [A] Qwen-2.5-7B-Instruct, quantifié 4 bits
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)                  # mode « réponse » (plus rapide)
print(f"Mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")

## 4. Génération des réponses (sauvegarde au fur et à mesure)

In [ ]:
import json
from src.evaluation import generate_answers

STAGE = "baseline"                                   # étape 4 : "finetuned"
pred_path = f"{EVAL_DIR}/{STAGE}_predictions.jsonl"

with open("data/final/test.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f]

done = []
if os.path.exists(pred_path):                        # reprise après déconnexion
    with open(pred_path, encoding="utf-8") as f:
        done = [json.loads(line) for line in f]
print(f"{len(done)} réponses déjà faites, {len(test) - len(done)} à faire")

for start in range(len(done), len(test), 10):        # par paquets de 10, sauvegardés tout de suite
    batch = generate_answers(model, tokenizer, test[start:start + 10], log_every=100)
    with open(pred_path, "a", encoding="utf-8") as f:
        for r in batch:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"{start + len(batch)}/{len(test)} sauvegardées")

## 5. Scores

In [ ]:
import pandas as pd
from src.evaluation import score_all

with open(pred_path, encoding="utf-8") as f:
    results = [json.loads(line) for line in f]
results, summary = score_all(results)
summary["memoire_gpu_max_go"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)

with open(f"{EVAL_DIR}/{STAGE}_scores.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
pd.Series(summary, name=STAGE).to_frame()

## 6. Quelques réponses à lire (une de chaque type)

In [ ]:
for family in ["service3", "medmcqa", "medquad"]:
    r = next(x for x in results if x["family"] == family)
    print(f"===== {family} =====\nQUESTION :\n{r['question'][:500]}\n\nRÉPONSE DU MODÈLE :\n{r['prediction'][:700]}"
          f"\n\nRÉFÉRENCE :\n{r['reference'][:400]}\n\nSCORES : {r['scores']}\n")

## 7. Enregistrement dans MLflow

In [ ]:
import mlflow
from src.evaluation import MAX_NEW_TOKENS

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("cardioagent-llm")
with mlflow.start_run(run_name=f"eval_{STAGE}"):
    mlflow.set_tags({"stage": STAGE, "model": MODEL_NAME, "family": "evaluation"})
    mlflow.log_params({"n_test": len(results), "decoding": "greedy",
                       **{f"max_new_tokens_{k}": v for k, v in MAX_NEW_TOKENS.items()}})
    mlflow.log_metrics(summary)
    mlflow.log_artifact(pred_path)
    mlflow.log_artifact(f"{EVAL_DIR}/{STAGE}_scores.json")
print("Run MLflow enregistré :", f"eval_{STAGE}")